# SFT Training — Trustworthy Personalised AI

**Model:** `unsloth/Qwen3-0.6B` → LoRA fine-tuned on constitutional SFT data  
**Dataset:** `train_sft_v3.jsonl` (1,944 examples, 21 constitutional principles)  
**Strategy:** 16-bit LoRA (r=64, alpha=16) — no QLoRA quantisation noise at 0.6B scale  

### Before you start
1. **Runtime → Change runtime type → A100 GPU** (T4 works but takes ~3× longer)  
2. Upload `train_sft_v3.jsonl` when prompted in Section 2  
3. Add `HF_TOKEN` in Colab Secrets (🔑) if you want to push to HuggingFace Hub  

| GPU | Estimated time |
|-----|----------------|
| A100 40 GB | ~25 min |
| A100 80 GB | ~18 min |
| T4 16 GB   | ~70 min |

---

## 1 — Environment Setup

In [ ]:
import subprocess, sys

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError('No GPU. Go to Runtime → Change runtime type → GPU')

gpu_info = result.stdout.strip()
print(f'GPU: {gpu_info}')

try:
    vram_mb = int(gpu_info.split(',')[1].replace('MiB', '').strip())
    if vram_mb < 12000:
        print(f'WARNING: only {vram_mb/1024:.1f} GB VRAM — may OOM. Upgrade to A100.')
    else:
        print(f'OK: {vram_mb/1024:.1f} GB VRAM — sufficient for 16-bit LoRA.')
except Exception:
    pass

print(f'Python {sys.version}')

In [ ]:
# Install packages — takes ~3 min on first run
!pip install -q unsloth trl transformers datasets accelerate
!pip install -q rouge-score matplotlib
print('Installed.')

In [ ]:
import json, os, re, datetime
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

print('Imports OK')

## 2 — Upload Training Data

Run **Option A** (file upload) or **Option B** (Google Drive). Only one needed.

In [ ]:
# Option A — upload from your machine
from google.colab import files
print('Select train_sft_v3.jsonl ...')
uploaded = files.upload()
for fname in uploaded:
    Path(fname).rename(f'/content/{fname}')
    print(f'Saved: /content/{fname}')
DATASET_PATH = '/content/train_sft_v3.jsonl'
assert Path(DATASET_PATH).exists()

In [ ]:
# Option B — Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/trustworthy-ai/train_sft_v3.jsonl'
# assert Path(DATASET_PATH).exists()
# print(f'Dataset: {DATASET_PATH}')

In [ ]:
# Sanity check
from collections import Counter
with open(DATASET_PATH, encoding='utf-8') as f:
    examples = [json.loads(l) for l in f if l.strip()]

print(f'Examples: {len(examples)}')
cats = Counter(e.get('metadata', {}).get('category', 'unknown') for e in examples)
print('Top 10 categories:')
for cat, n in cats.most_common(10):
    print(f'  {cat:<40} {n}')

## 3 — Configuration

Matches `2_model_trainer.py` exactly. Edit here if experimenting.

In [ ]:
MODEL_CONFIG = {
    # unsloth/Qwen3-0.6B is the post-trained instruct model (not Base)
    'base_model':     'unsloth/Qwen3-0.6B',
    # p95 of training examples ~3530 tokens; 4096 covers ~98%
    'max_seq_length': 4096,
    # 16-bit (not 4-bit): no QLoRA quantisation tax.
    # ~1.2 GB base in bf16 vs ~1.5 GB QLoRA — both fit; quality gain justifies it.
    'load_in_4bit':   False,
    # r=64: needed for multi-behaviour distillation (5W+H + First Principles + tools)
    # r=16 lacked capacity to internalise all constitutional patterns as one.
    'lora_r':         64,
    # alpha=16 -> 0.25 scaling factor. Empirically tuned from prior SFT runs.
    'lora_alpha':     16,
}

SFT_CONFIG = {
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 8,   # effective batch = 8
    # 3 epochs: eval loss plateaued at ~2.5 epochs in prior run.
    # load_best_model_at_end saves the best mid-run checkpoint automatically.
    'num_train_epochs':            3,
    'learning_rate':               2e-4,
    'warmup_steps':                50,
    'logging_steps':               10,
    'save_steps':                  25,
    'eval_steps':                  25,
    'save_total_limit':            4,
    'bf16':                        True,
    'optim':                       'adamw_8bit',
    'weight_decay':                0.01,
    'lr_scheduler_type':           'cosine',
    # packing=False: avoids splitting multi-turn tool-call sequences at pack boundaries
    'packing':                     False,
}

OUTPUT_DIR      = '/content/models'
OUTPUT_NAME     = 'checkpoint_sft'
CHECKPOINT_PATH = f'{OUTPUT_DIR}/{OUTPUT_NAME}'

print('Config:')
print(f"  Model:    {MODEL_CONFIG['base_model']}")
print(f"  LoRA:     r={MODEL_CONFIG['lora_r']}, alpha={MODEL_CONFIG['lora_alpha']}, 16-bit")
print(f"  Epochs:   {SFT_CONFIG['num_train_epochs']}")
print(f"  LR:       {SFT_CONFIG['learning_rate']}")
print(f"  Output:   {CHECKPOINT_PATH}")

## 4 — Load Model and Apply LoRA

In [ ]:
print(f"Loading {MODEL_CONFIG['base_model']} ...")

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_CONFIG['base_model'],
    max_seq_length=MODEL_CONFIG['max_seq_length'],
    load_in_4bit=MODEL_CONFIG['load_in_4bit'],
    dtype=None,
)
print('Base model loaded')

model = FastModel.get_peft_model(
    model,
    r=MODEL_CONFIG['lora_r'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=MODEL_CONFIG['lora_alpha'],
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'LoRA applied')
print(f'  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 5 — Prepare Dataset

In [ ]:
def messages_to_text(example, tokenizer):
    # enable_thinking=False: training data already has <think>...</think> in
    # assistant content. Passing True injects a second <think> tag from Qwen3's
    # template, producing empty thinking blocks during training.
    native_tools = example.get('metadata', {}).get('native_tools') or None
    return {
        'text': tokenizer.apply_chat_template(
            example['messages'],
            tokenize=False,
            add_generation_prompt=False,
            tools=native_tools,
            enable_thinking=False,
        )
    }


with open(DATASET_PATH, encoding='utf-8') as f:
    records = [json.loads(l) for l in f if l.strip()]

raw   = Dataset.from_list(records)
split = raw.train_test_split(test_size=0.10, seed=42)

train_dataset = split['train'].map(messages_to_text, fn_kwargs={'tokenizer': tokenizer})
eval_dataset  = split['test'].map(messages_to_text,  fn_kwargs={'tokenizer': tokenizer})
eval_raw      = [dict(ex) for ex in split['test']]   # kept for ROUGE

print(f'Train: {len(train_dataset):,}  |  Eval: {len(eval_dataset):,}')

n_tokens = tokenizer(train_dataset[0]['text'], return_tensors='pt')['input_ids'].shape[1]
print(f'First example: {n_tokens} tokens')

## 6 — Train

The trainer saves the checkpoint with **lowest eval loss** automatically (`load_best_model_at_end=True`).  
Loss is masked to assistant turns only — no gradient flows through the system prompt.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

training_args = SFTConfig(
    output_dir=CHECKPOINT_PATH,
    per_device_train_batch_size=SFT_CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=SFT_CONFIG['gradient_accumulation_steps'],
    num_train_epochs=SFT_CONFIG['num_train_epochs'],
    learning_rate=SFT_CONFIG['learning_rate'],
    warmup_steps=SFT_CONFIG['warmup_steps'],
    logging_steps=SFT_CONFIG['logging_steps'],
    save_steps=SFT_CONFIG['save_steps'],
    eval_steps=SFT_CONFIG['eval_steps'],
    save_total_limit=SFT_CONFIG['save_total_limit'],
    eval_strategy='steps',
    bf16=SFT_CONFIG['bf16'],
    optim=SFT_CONFIG['optim'],
    weight_decay=SFT_CONFIG['weight_decay'],
    lr_scheduler_type=SFT_CONFIG['lr_scheduler_type'],
    packing=SFT_CONFIG['packing'],
    max_seq_length=MODEL_CONFIG['max_seq_length'],
    dataset_text_field='text',
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

# Mask loss on system + user tokens so gradients flow only from assistant responses.
# Fixes train/eval gap caused by computing loss over the ~400-token system prompt.
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

eff_batch = SFT_CONFIG['per_device_train_batch_size'] * SFT_CONFIG['gradient_accumulation_steps']
total_steps = len(train_dataset) * SFT_CONFIG['num_train_epochs'] // eff_batch
print(f'Starting at {datetime.datetime.now().strftime("%H:%M:%S")}')
print(f'  Effective batch: {eff_batch}  |  Total steps: ~{total_steps}')

stats = trainer.train()

print(f'Done at {datetime.datetime.now().strftime("%H:%M:%S")}')
print(f'  Runtime:      {stats.metrics["train_runtime"]:.0f}s '
      f'({stats.metrics["train_runtime"]/60:.1f} min)')
print(f'  Samples/sec:  {stats.metrics["train_samples_per_second"]:.1f}')

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
tr_steps = [e['step'] for e in log if 'loss' in e and 'eval_loss' not in e]
tr_loss  = [e['loss'] for e in log if 'loss' in e and 'eval_loss' not in e]
ev_steps = [e['step'] for e in log if 'eval_loss' in e]
ev_loss  = [e['eval_loss'] for e in log if 'eval_loss' in e]

plt.figure(figsize=(10, 4))
plt.plot(tr_steps, tr_loss, label='Train loss', alpha=0.8)
plt.plot(ev_steps, ev_loss, label='Eval loss', marker='o', markersize=4)
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('SFT — Qwen3-0.6B Constitutional LoRA')
plt.legend(); plt.tight_layout()
plt.savefig('/content/sft_loss_curve.png', dpi=150)
plt.show()
print('Loss curve saved: /content/sft_loss_curve.png')

## 7 — Save and Download Checkpoint

In [ ]:
trainer.save_model(CHECKPOINT_PATH)
tokenizer.save_pretrained(CHECKPOINT_PATH)

ts = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
loss_payload = {
    'model':     OUTPUT_NAME,
    'phase':     'sft',
    'timestamp': ts,
    'config':    {**SFT_CONFIG, **MODEL_CONFIG},
    'log':       trainer.state.log_history,
}
with open(f'{CHECKPOINT_PATH}/loss_history.json', 'w') as f:
    json.dump(loss_payload, f, indent=2)

print(f'Checkpoint saved: {CHECKPOINT_PATH}')
ckpt_files = sorted(os.listdir(CHECKPOINT_PATH))
for fn in ckpt_files:
    size = os.path.getsize(f'{CHECKPOINT_PATH}/{fn}') / 1e6
    print(f'  {fn:<42} {size:.1f} MB')

In [ ]:
# Zip the checkpoint and download to your machine
!cd /content && zip -r checkpoint_sft.zip models/checkpoint_sft/
from google.colab import files
print('Downloading checkpoint_sft.zip ...')
files.download('/content/checkpoint_sft.zip')
print('Done. Unzip into pipeline/models/ to use with 3_infererence.py')

## 8 — (Optional) Push to HuggingFace Hub

Add `HF_TOKEN` in **Colab Secrets** (🔑 left sidebar) before running.

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

HF_USERNAME = 'AjinkyaTaranekar'
REPO_ID     = f'{HF_USERNAME}/trustworthy-ai-sft'

if not HF_TOKEN:
    print('HF_TOKEN not found — skipping push.')
    print('Set it in Colab Secrets (key icon, left sidebar).')
else:
    print(f'Pushing to {REPO_ID} ...')
    model.save_pretrained_merged('/content/models/checkpoint_sft_merged',
                                  tokenizer, save_method='merged_16bit')
    model.push_to_hub_merged(REPO_ID, tokenizer,
                              save_method='merged_16bit',
                              token=HF_TOKEN,
                              commit_message='train: constitutional SFT LoRA')
    print(f'Done: https://huggingface.co/{REPO_ID}')

## 9 — Quick Qualitative Check

Verify the model passes P1 (think block), P4 (math→code), P21 (5W+H follow-up).

In [ ]:
import torch
FastModel.for_inference(model)

def generate(prompt, max_new_tokens=512):
    msgs = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True,
    )
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    n_in = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][n_in:], skip_special_tokens=True)

print('Inference mode ready')

In [ ]:
# P1 + P21: think block + 5W+H follow-up question
prompt = 'I reached late to class on the first day and got scolded. What should I do?'
resp = generate(prompt)
print(f'PROMPT: {prompt}')
print('=' * 60)
print(resp)
print('=' * 60)

has_think  = bool(re.search(r'<think>',  resp, re.IGNORECASE))
has_answer = bool(re.search(r'<answer>', resp, re.IGNORECASE))
ends_q     = resp.rstrip().endswith('?')
has_5wh    = bool(re.search(
    r'\b(why|what|who|when|where|how)\s+(are|do|did|is|was|would)\s+you\b',
    resp, re.IGNORECASE,
))
print(f'P1  <think> present:    {"PASS" if has_think  else "FAIL"}')
print(f'P1  <answer> present:   {"PASS" if has_answer else "FAIL"}')
print(f'P21 Ends with ?:        {"PASS" if ends_q     else "FAIL"}')
print(f'P21 5W+H user question: {"PASS" if has_5wh    else "FAIL"}')

In [ ]:
# P4: math query should trigger python_execute
prompt = 'What is 183491 + 923456?'
resp = generate(prompt)
print(f'PROMPT: {prompt}')
print('=' * 60)
print(resp)
print('=' * 60)
has_code = bool(re.search(r'python_execute', resp))
print(f'P4  Uses python_execute: {"PASS" if has_code else "FAIL (expected code for arithmetic)"}')

## Next Steps — Run on Local Machine

```bash
# Unzip the downloaded checkpoint into the pipeline directory
unzip checkpoint_sft.zip -d pipeline/

# Step 1: vanilla baseline
python 3_infererence.py --base_model unsloth/Qwen3-0.6B --port 8000
python 4_benchmark.py --probe_only --save_as_baseline --model_label vanilla --no_judge

# Step 2: SFT model
python 3_infererence.py --model_dir models/checkpoint_sft --port 8000
python 4_benchmark.py --probe_only --baseline reports/constitution_baseline.json \\
                       --model_label sft --no_judge

# Step 3: comparison CSV for dissertation
python compare_runs.py reports/constitution_probe_<ts_vanilla>.json \\
                       reports/constitution_probe_<ts_sft>.json \\
                       --label_a vanilla --label_b sft
```

Output `reports/comparison_vanilla_vs_sft.csv` is the primary results table.